In [13]:
import google.generativeai as genai
import pandas as pd

In [4]:
API_KEY = "YOUR_API_KEY"

genai.configure(api_key=API_KEY)

model = genai.GenerativeModel('gemini-3-flash-preview')

response = model.generate_content("Say hello as a marketing analyst in one sentence")
print(response.text)

Hello, I’ve analyzed the latest performance metrics and am ready to translate those data-driven insights into a high-ROI strategy for your brand.


In [6]:
df = pd.read_csv('marketing_campaigns.csv')
print(df.shape)
print(df.head())

(200, 12)
  campaign_id            campaign_name campaign_type target_audience    month  \
0        C001       Display Campaign 1       Display           45-54  2024-04   
1        C002    Influencer Campaign 2    Influencer             55+  2024-02   
2        C003  Social Media Campaign 3  Social Media           45-54  2024-11   
3        C004    Influencer Campaign 4    Influencer           18-24  2024-08   
4        C005    Influencer Campaign 5    Influencer           18-24  2024-08   

   budget  spend  impressions  clicks  leads  conversions  revenue  
0   22642  17913        75124    2383    197           37    75708  
1   23444  20221       120530   12032   5043         2415    36771  
2    9966   7824        56924    4844   1407           89    41078  
3   13367  11263        80621    9341     82           32    19797  
4   19995  19136       131294    1040    203           21    34720  


In [7]:
summary = df.groupby('campaign_type').agg(
    total_revenue = ('revenue', 'sum'),
    total_spend = ('spend', 'sum'),
    total_clicks = ('clicks', 'sum'),
    total_conversions = ('conversions', 'sum'),
    total_leads = ('leads', 'sum'),
    num_campaigns = ('campaign_id', 'count')
).reset_index()

summary['roi_pct'] = round((summary['total_revenue'] - summary['total_spend']) / summary['total_spend'] * 100, 2)
summary['conversion_rate'] = round(summary['total_conversions'] / summary['total_clicks'] * 100, 2)
summary['cost_per_conversion'] = round(summary['total_spend'] / summary['total_conversions'], 2)

print(summary)

  campaign_type  total_revenue  total_spend  total_clicks  total_conversions  \
0       Display        1306057       449992        178686               6098   
1         Email        1101641       372457        163714              10308   
2    Influencer        1685214       547054        223981              13580   
3        Search        1948309       634497        197894              14980   
4  Social Media        1529833       452565        208787              15300   

   total_leads  num_campaigns  roi_pct  conversion_rate  cost_per_conversion  
0        29251             36   190.24             3.41                73.79  
1        30121             36   195.78             6.30                36.13  
2        47085             40   208.05             6.06                40.28  
3        43506             48   207.06             7.57                42.36  
4        50252             40   238.04             7.33                29.58  


In [10]:
from datetime import date
today = date.today()

prompt = f"""
You are a senior marketing analyst.
Today's date is {today}.
Analyze this campaign performance data and write a professional report:
{summary.to_string()}
Your report must include:
1. Executive Summary
2. Best performing campaign type and why
3. Worst performing campaign type and why
4. 3 specific actionable recommendations
5. Budget allocation suggestion

Write it professionally as if presenting to a CEO.
"""

response = model.generate_content(prompt)
print(response.text)

**To:** Chief Executive Officer
**From:** Senior Marketing Analyst
**Date:** March 14, 2026
**Subject:** Q1 2026 Marketing Campaign Performance Analysis and Strategic Recommendations

---

### 1. Executive Summary
This report provides a comprehensive analysis of our marketing performance across five primary channels: Display, Email, Influencer, Search, and Social Media. Overall, the marketing portfolio is performing strongly with an average ROI across all channels exceeding 200%. We generated a total revenue of $7.57M against a total spend of $2.46M. 

While **Search** remains our primary revenue driver, **Social Media** has emerged as the most efficient channel in terms of Return on Investment (ROI) and cost-efficiency. Conversely, the **Display** channel is currently underperforming relative to the portfolio, characterized by high acquisition costs and the lowest conversion rates.

---

### 2. Best Performing Campaign Type: Social Media
**Social Media** is unequivocally the top-perfo

In [12]:
report_text = response.text

with open('marketing_ai_report.txt', 'w') as f:
    f.write("MARKETING CAMPAIGN ANALYSIS REPORT\n")
    f.write("Generated by AI Marketing Agent\n")
    f.write("="*50 + "\n\n")
    f.write(report_text)

print("Report saved! ✅")

Report saved! ✅
